In [1]:
!pip install -q sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 61.4 MB/s eta 0:00:00


In [2]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Your "documents" — think of these as pages in a knowledge base
documents = [
    "Python lists are ordered, mutable collections. You can add items with append() and remove with remove().",
    "Python dictionaries store key-value pairs. Access values with dict['key'] or dict.get('key').",
    "A for loop in Python iterates over a sequence. Use 'for item in list' to loop through items.",
    "Functions in Python are defined with 'def'. They take parameters and return values using 'return'.",
    "Python classes use 'class' keyword. '__init__' is the constructor that runs when an object is created.",
    "List comprehensions create lists in one line: [x*2 for x in range(10)] doubles each number.",
    "Try-except blocks handle errors in Python. Put risky code in 'try' and handle errors in 'except'.",
    "Python imports let you use external libraries. Use 'import library' or 'from library import function'.",
]

# Load the embedding model — turns text into meaning-based numbers
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Embed all documents — convert each to a list of numbers
doc_embeddings = embedder.encode(documents)

# Build the search index (FAISS)
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings))

print(f"Documents embedded: {len(documents)}")
print(f"Each embedding size: {dimension} numbers")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Documents embedded: 8
Each embedding size: 384 numbers


In [3]:
def retrieve(question, top_k=2):
    # Turn the question into numbers (same embedding space as documents)
    question_embedding = embedder.encode([question])

    # Search for the closest documents in that space
    distances, indices = index.search(np.array(question_embedding), top_k)

    # Return the matching documents
    results = [documents[i] for i in indices[0]]
    return results

# Test it with a question
question = "How do I loop through a list?"
retrieved = retrieve(question)

print(f"Question: {question}")
print(f"\nRetrieved documents:")
for i, doc in enumerate(retrieved):
    print(f"{i+1}. {doc}")

Question: How do I loop through a list?

Retrieved documents:
1. A for loop in Python iterates over a sequence. Use 'for item in list' to loop through items.
2. List comprehensions create lists in one line: [x*2 for x in range(10)] doubles each number.


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load the same small model as before
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float16, device_map="auto"
)

def rag_answer(question):
    # Step 1: Retrieve relevant documents
    relevant_docs = retrieve(question, top_k=2)
    context = "\n".join(relevant_docs)

    # Step 2: Build a grounded prompt
    prompt = f"""Answer the question using ONLY the information below.
If the answer isn't in the information, say "I don't know."

Information:
{context}

Question: {question}
Answer:"""

    # Step 3: Generate answer from the model
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=150, do_sample=False)
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    print(f"Question: {question}")
    print(f"\nContext retrieved:\n{context}")
    print(f"\nAnswer: {response}")
    print("-"*60)

# Test with two questions
rag_answer("How do I loop through a list?")
rag_answer("What is a constructor in Python?")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Question: How do I loop through a list?

Context retrieved:
A for loop in Python iterates over a sequence. Use 'for item in list' to loop through items.
List comprehensions create lists in one line: [x*2 for x in range(10)] doubles each number.

Answer: Use 'for item in list' to loop through items.
------------------------------------------------------------
Question: What is a constructor in Python?

Context retrieved:
Python classes use 'class' keyword. '__init__' is the constructor that runs when an object is created.
Functions in Python are defined with 'def'. They take parameters and return values using 'return'.

Answer: A constructor in Python is the method defined within a class that gets automatically called when an instance of the class is created. It serves to initialize the attributes of the new object. The name of this method is typically prefixed with `__` (double underscore) to make it private and prevent direct access from outside the class.
----------------------------

In [5]:
# Test with a question NOT in your documents
rag_answer("How do I use async/await in Python?")

Question: How do I use async/await in Python?

Context retrieved:
Functions in Python are defined with 'def'. They take parameters and return values using 'return'.
Try-except blocks handle errors in Python. Put risky code in 'try' and handle errors in 'except'.

Answer: I don't know.
------------------------------------------------------------
